In [1]:
# ============================================================
#  CLASE: WORKING WITH LLMs - API DE OPENAI
#  Rol: Analista de Datos e IA
#  Versión didáctica para alumnos principiantes
# ============================================================

from openai import OpenAI
import getpass
import json

# ------------------------------------------------------------
#  CONFIGURACIÓN
# ------------------------------------------------------------
api_key = getpass.getpass("🔑 Ingresa tu API KEY de OpenAI: ")
client = OpenAI(api_key=api_key)
print(" Listo!\n")

# Definimos UN solo rol que se reutiliza en todos los ejemplos
ROL = "Eres un analista de datos amable que explica resultados de forma clara y simple."

# Datos de ejemplo: ventas mensuales de una pequeña tienda
DATOS = """
Ventas del último trimestre:
- Enero:   $12.000
- Febrero: $9.500
- Marzo:   $15.200
"""


🔑 Ingresa tu API KEY de OpenAI:  ········


 Listo!



In [2]:

# ============================================================
#  BLOQUE 1: PRIMERA LLAMADA A LA API
# ------------------------------------------------------------
#  Conceptos: model, system prompt, user prompt, temperature
# ============================================================
print("=" * 50)
print("  BLOQUE 1: Tu primera llamada al LLM")
print("=" * 50)

respuesta = client.chat.completions.create(
    model="gpt-4o-mini",                 # Modelo: rápido y económico
    messages=[
        {"role": "system", "content": ROL},        # Define cómo se comporta
        {"role": "user", "content": f"Analiza estas ventas:\n{DATOS}"}
    ],
    temperature=0.3,                     # 0 = preciso | 1 = creativo
    max_tokens=200,                      # Largo máximo de la respuesta
)

print(respuesta.choices[0].message.content)
print(f"\n📊 Tokens usados: {respuesta.usage.total_tokens}")




  BLOQUE 1: Tu primera llamada al LLM
Claro, analicemos las ventas del último trimestre.

1. **Total de Ventas**: Primero, sumemos las ventas de los tres meses:
   - Enero: $12,000
   - Febrero: $9,500
   - Marzo: $15,200

   Total = $12,000 + $9,500 + $15,200 = **$36,700**

2. **Promedio Mensual**: Ahora, calculemos el promedio de ventas por mes:
   - Promedio = Total de Ventas / Número de Meses
   - Promedio = $36,700 / 3 ≈ **$12,233.33**

3. **Análisis Mensual**:
   - **Enero**: $12,000, que es un buen inicio de trimestre.
   - **Febrero**: $9,500, que es una caída en comparación con enero. Esto podría indicar una baja estacional o un

📊 Tokens usados: 265



⏎ Enter para continuar... 


''

In [3]:
# ============================================================
#  BLOQUE 2: EFECTO DE TEMPERATURE
# ------------------------------------------------------------
#  Mismo prompt, distinta temperature → distinto estilo
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 2: Cómo cambia la respuesta con temperature")
print("=" * 50)

pregunta = "Sugiere un nombre creativo para un dashboard de ventas."

for temp in [0, 1.5]:
    print(f"\n--- temperature = {temp} ---")
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ROL},
            {"role": "user", "content": pregunta}
        ],
        temperature=temp,
        max_tokens=60,
    )
    print(r.choices[0].message.content)

print("\n💡 temperature baja = consistente | temperature alta = creativo")




  BLOQUE 2: Cómo cambia la respuesta con temperature

--- temperature = 0 ---
¡Claro! Aquí tienes algunas sugerencias creativas para un nombre de dashboard de ventas:

1. **"Ventas en Vista"**
2. **"Rumbo a las Ventas"**
3. **"El Radar de Ventas"**
4. **"Ventas al Vuelo"**


--- temperature = 1.5 ---
¡Claro! Aquí tienes algunas ideas creativas para un nombre de dashboard de ventas:

1. **Ventas en Vanguardia**
2. **Impulso Óptimo**
3. **Radar de Rentabilidad**
4. **Vista Vertiente: Rendimiento en Tiempo Real**
5. **

💡 temperature baja = consistente | temperature alta = creativo



⏎ Enter para continuar... 


''

In [4]:

# ============================================================
#  BLOQUE 3: SALIDA EN JSON (para usar en un sistema)
# ------------------------------------------------------------
#  Cuando queremos integrar el LLM con código o una BD
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 3: Respuesta en formato JSON")
print("=" * 50)

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": ROL + " Devuelve SIEMPRE un JSON con: tendencia, mes_mayor, mes_menor, recomendacion."
        },
        {"role": "user", "content": DATOS}
    ],
    temperature=0,
    response_format={"type": "json_object"},
    max_tokens=200,
)

datos = json.loads(r.choices[0].message.content)
print(json.dumps(datos, indent=2, ensure_ascii=False))
print("\n💡 Este JSON ya se puede guardar en una base de datos o mostrar en un dashboard.")






  BLOQUE 3: Respuesta en formato JSON
{
  "tendencia": "Aumento",
  "mes_mayor": "Marzo",
  "mes_menor": "Febrero",
  "recomendacion": "Considerar estrategias para mantener el crecimiento en marzo y analizar las causas de la baja en febrero."
}

💡 Este JSON ya se puede guardar en una base de datos o mostrar en un dashboard.


In [ ]:

# ============================================================
#  BLOQUE 4: CONVERSACIÓN CON MEMORIA (multi-turno)
# ------------------------------------------------------------
#  El LLM no recuerda nada → tenemos que enviarle el historial
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 4: Conversación con contexto")
print("=" * 50)
print("Escribe tus preguntas. Escribe 'salir' para terminar.\n")

historial = [
    {"role": "system", "content": ROL},
    {"role": "user", "content": f"Te paso estos datos:\n{DATOS}"}
]

# Primera respuesta del bot para arrancar
r = client.chat.completions.create(
    model="gpt-4o-mini", messages=historial, temperature=0.3, max_tokens=150
)
historial.append({"role": "assistant", "content": r.choices[0].message.content})
print(f"🤖 Analista: {r.choices[0].message.content}\n")

# Loop de conversación
while True:
    pregunta = input("👤 Tú: ").strip()
    if pregunta.lower() == "salir":
        break
    historial.append({"role": "user", "content": pregunta})

    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=historial, temperature=0.3, max_tokens=200
    )
    respuesta = r.choices[0].message.content
    historial.append({"role": "assistant", "content": respuesta})
    print(f"🤖 Analista: {respuesta}\n")

print(f"\n Cada turno reenvía TODO el historial. Por eso decimos que el LLM es 'sin memoria'.")
print("\n ¡Fin de la clase! Ya viste los 4 conceptos clave para trabajar con LLMs.")


  BLOQUE 4: Conversación con contexto
Escribe tus preguntas. Escribe 'salir' para terminar.

🤖 Analista: ¡Claro! Vamos a analizar las ventas del último trimestre.

1. **Total de Ventas**: Primero, sumemos las ventas de los tres meses:
   - Enero: $12,000
   - Febrero: $9,500
   - Marzo: $15,200

   Total = $12,000 + $9,500 + $15,200 = **$36,700**

2. **Promedio Mensual**: Para encontrar el promedio de ventas por mes, dividimos el total entre 3:
   - Promedio = Total / 3 = $36,700 / 3 ≈ **$12,233.33**

3. **Variación Mensual**: Ahora, veamos



👤 Tú:  subieron las ventas para el mes de marzo?


🤖 Analista: Sí, las ventas subieron en marzo en comparación con los meses anteriores. Aquí tienes un resumen de la variación:

- **Enero**: $12,000
- **Febrero**: $9,500 (disminución respecto a enero)
- **Marzo**: $15,200 (aumento respecto a febrero)

Para ser más específicos:
- De enero a febrero, las ventas bajaron en $2,500.
- De febrero a marzo, las ventas aumentaron en $5,700.

Así que sí, marzo tuvo un buen incremento en ventas en comparación con febrero. ¡Eso es una buena noticia!



👤 Tú:  0


🤖 Analista: Parece que escribiste "0". Si necesitas más información o tienes otra pregunta sobre las ventas o cualquier otro tema, ¡estaré encantado de ayudarte!

